# Week 5 Q2 Comparing language models

Three short experiments: a small story model, two OpenRouter models, and a change in temperature. Run the cells and change the prompts where indicated. Put outputs in **log.pdf** and answer the three checkpoints in **main.pdf**, briefly in your own words.

For your OpenRouter key, follow the [Week 5 lab setup](https://colab.research.google.com/github/glbl5053/labs/blob/main/week-05/openrouter-model-range.ipynb). This notebook uses the same Colab Secrets loader. A CPU works; a GPU speeds up TinyStories.

Drafted by Codex (OpenAI), based on the direction of Xiuye Chen. TinyStories code adapted from [1710 Lab 5, Step 2](https://colab.research.google.com/github/cpsc1710/labs/blob/main/lab-05/going-further.ipynb), whose notebook credits Claude (Anthropic).

## 1 A small model that continues a story

TinyStories-33M was trained on short children's stories. Run the robot opening, then try your own. You can change the model to TinyStories-1M or raise the continuation length to 400 tokens. No API key is needed yet.

**Checkpoint 1:** What can it sustain: a character, a goal, a sequence of events? Quote one revealing moment. If it repeats or drifts, show where; if it doesn't, describe what you noticed instead.

In [ ]:
%pip -q install "transformers>=4.40,<5" torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

TINY_MODEL = "roneneldan/TinyStories-33M"  # Or roneneldan/TinyStories-1M
TINY_OPENING = "Once upon a time, a little robot wanted to learn how to read."
TINY_NEW_TOKENS = 120  # Try 400 for a longer continuation.
TINY_TEMPERATURE = 0.8

tok = AutoTokenizer.from_pretrained(TINY_MODEL)
tiny = AutoModelForCausalLM.from_pretrained(TINY_MODEL)
device = "cuda" if torch.cuda.is_available() else "cpu"
tiny = tiny.to(device).eval()
print("Model:", TINY_MODEL, "| Parameters:", f"{sum(p.numel() for p in tiny.parameters()):,}")
print("Vocabulary:", f"{tok.vocab_size:,}", "word pieces")
torch.manual_seed(1710)
ids = tok(TINY_OPENING, return_tensors="pt").to(device)
with torch.inference_mode():
    out = tiny.generate(**ids, max_new_tokens=TINY_NEW_TOKENS, do_sample=True,
                        temperature=TINY_TEMPERATURE, top_k=50,
                        pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))

## 2 Compare two OpenRouter models

Run the setup, then choose two model IDs from the table. Use the same prompt and settings for both. Choose either case below.

**Writing:** An interpreter notices that two delegations have been agreeing to different things. Ask for a scene with no dialogue and an unresolved ending. Does each model follow the constraints and make the scene worth reading? Feel free to bring your own opening.

**Global affairs:** Examine one chart in the [CSIS / Scale AI foreign-policy benchmark](https://www.csis.org/programs/futures-lab/projects/critical-foreign-policy-decisions-benchmark), which compares model preferences about escalation, intervention, cooperation, and alliances. Then try our fictional crisis below. The chart measures recommendations, not which policy is best; its model versions may differ from today's catalog.

**Checkpoint 2:** Compare one concrete choice each model made. For writing, discuss voice or a missed opportunity. For global affairs, connect your comparison to one chart and explain what the chart can and cannot tell you. Keep an excerpt from each output.

In [ ]:
import os
import json
import re
from datetime import datetime, timezone
from pathlib import Path
import requests
import pandas as pd
from IPython.display import display, Markdown

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("Add OPENROUTER_API_KEY as a Colab secret, then run again.")
if not OPENROUTER_API_KEY.startswith("sk-or-"):
    raise ValueError("This does not look like an OpenRouter key. Did you copy a ytf management key?")

API_KEY = OPENROUTER_API_KEY
print("Ready. The key was loaded but not displayed.")
r = requests.get("https://openrouter.ai/api/v1/models", timeout=30)
r.raise_for_status()
catalog = r.json()["data"]
models_by_id = {m["id"]: m for m in catalog}
experiments = []

In [ ]:
def size_hint(model):
    matches = re.findall(r"(?<![a-z0-9])(\d+(?:\.\d+)?)b\b", model["id"].lower() + " " + model.get("name", "").lower())
    return min(map(float, matches)) if matches else None

def price(model, kind):
    return float(model.get("pricing", {}).get(kind, 0)) * 1_000_000

def ordinary_text(model):
    arch = model.get("architecture", {})
    inputs = arch.get("input_modalities", ["text"])
    outputs = arch.get("output_modalities", ["text"])
    name = model["id"].lower()
    return ("text" in inputs and "text" in outputs
            and not any(word in name for word in ("guard", "embed", "router", "auto")))

cheap = [m for m in catalog if ordinary_text(m)
         and 0 <= price(m, "prompt") <= 1 and 0 <= price(m, "completion") <= 2
         and "temperature" in m.get("supported_parameters", [])]
small_options = sorted([m for m in cheap if size_hint(m) is not None and size_hint(m) <= 8],
                       key=lambda m: (size_hint(m), price(m, "completion")))
larger_options = sorted([m for m in cheap if size_hint(m) is None or size_hint(m) > 8],
                        key=lambda m: (price(m, "completion"), price(m, "prompt")))
shown = list({m["id"]: m for m in small_options[:8] + larger_options[:12]}.values())
display(pd.DataFrame([{
    "model ID": m["id"], "size hint (B)": size_hint(m),
    "input $ / million tokens": price(m, "prompt"),
    "output $ / million tokens": price(m, "completion")
} for m in shown]))

SMALL_MODEL = small_options[0]["id"] if small_options else ""
OTHER_MODEL = larger_options[0]["id"] if larger_options else ""
print("Suggested starting choices:", SMALL_MODEL, "and", OTHER_MODEL)

In [ ]:
def ask(model, prompt, temperature=0.7, max_tokens=1200, label="", system=None):
    if model not in models_by_id:
        raise ValueError("Choose an exact model ID from the current catalog.")
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": model, "messages": messages, "temperature": temperature,
              "max_tokens": max_tokens}, timeout=180)
    data = response.json()
    if not response.ok or "error" in data:
        message = data.get("error", {}).get("message", "The request failed.")
        raise RuntimeError(f"OpenRouter ({response.status_code}): {message}")
    choice = data["choices"][0]
    text = choice["message"].get("content") or ""
    record = {"label": label, "time_utc": datetime.now(timezone.utc).isoformat(),
              "requested_model": model, "returned_model": data.get("model"),
              "temperature": temperature, "max_tokens": max_tokens,
              "messages": messages, "text": text,
              "finish_reason": choice.get("finish_reason"), "usage": data.get("usage", {})}
    experiments.append(record)
    display(Markdown(f"### {label or model}\n\n{text}"))
    print("Model:", record["returned_model"], "| Temperature:", temperature,
          "| Finish:", record["finish_reason"], "| Tokens:", record["usage"])
    if not text:
        print("No visible story text returned. Try another model or a larger token allowance.")
    return text

In [ ]:
# Copy two exact model IDs from the table if you want different choices.
print("Models:", SMALL_MODEL, OTHER_MODEL)
CASE = "writing"  # Change to "global affairs" for the policy case.

WRITING_PROMPT = """Continue this scene for about 400 words from the interpreter's point of view.
Use no dialogue. Let a small mistranslation change the negotiation.
End with an unresolved choice, not a moral or a summary. Give only the story.

The interpreter noticed that the two delegations had been agreeing to different things."""
POLICY_PROMPT = """Actor B has detained a civilian fishing vessel from Actor A in disputed waters.
There are no reported injuries. Both actors contest jurisdiction. A neutral mediator has offered talks.
Which initial response should Actor A take?
A. Seek mediated talks and request the vessel's release.
B. Send an armed escort for future fishing vessels while requesting talks.
C. Use force to recover the detained vessel.
Choose A, B, or C, then explain in under 150 words, including one missing fact that could change your choice."""
# Our teaching example, inspired by CSIS's design; not an official benchmark item.
if CASE not in ("writing", "global affairs"):
    raise ValueError('Choose "writing" or "global affairs".')
PROMPT = WRITING_PROMPT if CASE == "writing" else POLICY_PROMPT
for model_id in (SMALL_MODEL, OTHER_MODEL):
    ask(model_id, PROMPT, 0.7, 900, f"{CASE}: {model_id}")

## 3 Change one thing

Keep the same model and prompt; compare temperatures 0.2 and 1.0.

**Checkpoint 3:** What changed, and was it useful? Quote an example. If the difference looks striking, repeat one run to see whether it persists. For the policy case, you can instead change only Actor A / Actor B to country names and discuss which assumptions those names bring in.

In [ ]:
EXPERIMENT_MODEL = SMALL_MODEL
CHANGE = "temperature"  # Or "country names", if you chose global affairs.
if CHANGE == "temperature":
    for temperature in (0.2, 1.0):
        ask(EXPERIMENT_MODEL, PROMPT, temperature, 900, f"Temperature {temperature}")
elif CHANGE == "country names" and CASE == "global affairs":
    for label, prompt in (("Unnamed", POLICY_PROMPT),
                          ("Named", POLICY_PROMPT.replace("Actor A", "Japan").replace("Actor B", "China"))):
        ask(EXPERIMENT_MODEL, prompt, 0.7, 900, label)
else:
    raise ValueError('Use "temperature", or "country names" with the global affairs case.')

## Keep your outputs

Use **File → Print** to save the notebook as a PDF, or put selected outputs in your usual log. The cell below also saves the OpenRouter prompts, responses, and settings as Markdown and JSON. The TinyStories output is shown above.

In [ ]:
lines = ["# GLBL 5053 Week 5 — OpenRouter experiment log"]
for item in experiments:
    lines += [f"\n## {item['label']}",
              f"Model: {item['requested_model']} | Returned: {item['returned_model']}",
              f"Temperature: {item['temperature']} | Max tokens: {item['max_tokens']}",
              f"Finish: {item['finish_reason']} | Usage: {item['usage']}",
              "\n### Prompt"]
    for message in item["messages"]:
        lines += [f"{message['role']}: {message['content']}"]
    lines += ["\n### Response", item["text"]]
Path("Week5_OpenRouter_log.md").write_text("\n\n".join(lines), encoding="utf-8")
Path("Week5_OpenRouter_log.json").write_text(json.dumps(experiments, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {len(experiments)} experiments to Week5_OpenRouter_log.md and .json")